# Ch 3. AI Assistant 시스템 개요

## 이 챕터에서 배우는 것

- 프로덕션 AI Assistant를 이루는 **8개의 블록** — 입력·이해·검색·생성·검증·저장·모니터링·휴먼 핸드오프
- 각 블록이 **코드·모델·RAG·외부시스템** 중 무엇으로 구현되는지 구분하기
- 자기 프로젝트의 Assistant 아키텍처를 **한 장의 다이어그램**으로 그리는 연습

원본 챕터: [AI Assistant 시스템 개요](https://desty.github.io/study-ai-assistant-engineering/part1/03-assistant-overview/)

In [ ]:
# 환경 설치
!pip install -q anthropic pandas

In [ ]:
import os
from getpass import getpass
os.environ.setdefault('ANTHROPIC_API_KEY', getpass('ANTHROPIC_API_KEY: '))

## 1. "Assistant = 프롬프트 하나" 가 아닙니다

많은 입문자가 이렇게 생각합니다:

```
사용자 메시지 → LLM 한 번 호출 → 답변
```

데모나 PoC는 가능. **운영에서는 거의 모든 케이스에서 실패**.

실전 Assistant는 **여러 블록이 협력**합니다.

## 2. 8개의 블록 — 전체 파이프라인

```
1. 입력(Intake) - 메시지·파일·세션 수신
   ↓
2. 이해(Understand) - 의도·엔티티 분류
   ↓
3. 검색(Retrieve) - 문서·DB·외부 API
   ↓
4. 생성(Generate) - LLM으로 답변 작성
   ↓
5. 검증(Validate) - 가드레일·안전성 검사
   ↓
6. 저장(Persist) - 대화로그·피드백 저장
   ↓
7. 모니터링(Observe) - 지연·비용·품질 추적
   ↓
8. 휴먼 핸드오프(Escalate) - 사람 담당자에게 이관
```

## 3. 각 블록 — 무엇이 들어있나

In [ ]:
import pandas as pd

blocks = [
    {
        "블록": "1. 입력",
        "주요 기능": "메시지 수신, 파일 처리",
        "구현": "코드",
        "핵심": "100% 코드 기반"
    },
    {
        "블록": "2. 이해",
        "주요 기능": "의도 분류, 엔티티 추출",
        "구현": "LLM",
        "핵심": "의도가 다음 결정 결정"
    },
    {
        "블록": "3. 검색",
        "주요 기능": "문서 검색, DB 조회",
        "구현": "RAG",
        "핵심": "품질 70% 결정"
    },
    {
        "블록": "4. 생성",
        "주요 기능": "답변 작성",
        "구현": "LLM",
        "핵심": "검색이 제대로 돼야 효과적"
    },
    {
        "블록": "5. 검증",
        "주요 기능": "안전성, 정책 준수",
        "구현": "LLM + 규칙",
        "핵심": "계층형 방어 필수"
    },
    {
        "블록": "6. 저장",
        "주요 기능": "대화로그, 피드백",
        "구현": "DB",
        "핵심": "자기 개선 루프의 기반"
    },
    {
        "블록": "7. 모니터링",
        "주요 기능": "지연, 비용, 품질",
        "구현": "대시보드",
        "핵심": "문제 조기 발견"
    },
    {
        "블록": "8. 핸드오프",
        "주요 기능": "사람 담당자에게 이관",
        "구현": "큐",
        "핵심": "자율성은 기본값 아님"
    },
]

df = pd.DataFrame(blocks)
print(df.to_string(index=False))

## 4. 코드 vs 모델 vs RAG vs 외부시스템 — 라벨링

8블록을 **기술 선택** 라벨로 분류하면:

- **CODE**: 결정론적 로직 (레이트리밋·권한·스키마 검증)
- **MODEL**: LLM 호출 (의도 분류·답변 생성)
- **RAG**: 임베딩+벡터검색 (회사 문서·지식베이스)
- **EXT**: DB·API·메시징 (실제 데이터·액션)

## 5. 간단한 예시 — 고객 문의 어시스턴트

"환불 관련 문의"를 처리하는 Assistant의 구체 흐름:

In [ ]:
from anthropic import Anthropic

client = Anthropic()

user_message = "지난주에 산 이어폰 환불하고 싶은데 어떻게 해요?"

print("=== 고객 문의 처리 흐름 ===")
print(f"\n사용자: {user_message}")
print()

# 1. 입력 단계
print("1) [입력] 레이트리밋 통과")
print()

# 2. 이해 단계
understand_prompt = f"""다음 메시지에서 의도를 분류하세요.
메시지: {user_message}
의도는 '환불 요청' 또는 '정보 요청' 중 하나입니다."""

r_understand = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=50,
    messages=[{"role": "user", "content": understand_prompt}],
)
print("2) [이해]")
print(r_understand.content[0].text)
print()

# 3. 검색 (시뮬레이션)
print("3) [검색] 환불 정책 문서 → 7일 이내 환불 가능")
print()

# 4. 생성
generate_prompt = f"""환불 정책에 따라 답변하세요.
정책: 구매 후 7일 이내 환불 가능
사용자: {user_message}
친절하게 2줄로 답변하세요."""

r_generate = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    system="당신은 고객 서비스 어시스턴트입니다.",
    messages=[{"role": "user", "content": generate_prompt}],
)
print("4) [생성]")
print(r_generate.content[0].text)
print()

# 5~8단계
print("5) [검증] 개인정보 필터 통과")
print("6) [저장] 대화 로그 저장")
print("7) [모니터링] 지연·비용 기록")
print("8) [핸드오프] 상담원 연결 준비")

## 6. 실전 튜토리얼 — 자기 Assistant 구조도 그리기

지금 만들고 싶은 Assistant를 정하고 아래 순서로 진행합니다:

### 단계 1. 유즈케이스 정의
- 이름: (예: "고객 환불 어시스턴트")
- 사용자: (예: "고객 서비스팀")
- 입력 예시 3개
- 기대 출력 예시 3개

### 단계 2. 8개 블록 중 사용할 것 선택
처음엔 **입력·이해·생성·저장** 넷만으로도 시작 가능.

### 단계 3. 각 블록에 라벨 붙이기
`CODE` · `MODEL` · `RAG` · `EXT` 중 무엇으로 구현할지.

### 단계 4. 다이어그램으로 그리기
노트·화이트보드 또는 Figma·draw.io 등에 손으로 그려보세요.

### 단계 5. 실패 시나리오 10개 쓰기
각 블록이 실패했을 때 어떻게 될지 예측하세요.

In [ ]:
failure_scenarios = [
    "의도 분류 실패 → 엉뚱한 문서 검색",
    "검색 실패 → 답변 생성 불가",
    "생성 중 hallucination → 거짓 정보",
    "개인정보 누설 → 보안 위협",
    "출력 형식 오류 → 파싱 실패",
    "외부 API 타임아웃 → 답변 지연",
    "DB 저장 실패 → 로그 손실",
    "모니터링 alert 미발생 → 문제 발견 지연",
    "핸드오프 미실행 → 사용자 좌절",
    "동시성 문제 → 중복 처리",
]

print("=== 실패 시나리오 10가지 ===")
for i, scenario in enumerate(failure_scenarios, 1):
    print(f"{i}. {scenario}")

## Part 1을 마치며

지금까지 배운 것:

| 챕터 | 배운 것 | 다음 단계 |
|---|---|---|
| 1 | 모델이 필요한 문제 판별 | 전체 모든 챕터에 적용 |
| 2 | LLM이 어떻게 작동하는가 | Part 2에서 API로 실전 |
| 3 | Assistant의 8블록 구조 | Part 2~7이 각 블록을 깊이 다룸 |

**Part 2부터는 코드로 손을 댑니다.** 첫 LLM 호출부터 시작해, 구조화 출력·툴 콜링·스트리밍까지.

## 다음

Part 2에서는 각 블록을 실제 코드로 구현합니다.

- Ch 4. OpenAI / Anthropic API 시작하기
- Ch 5. 구조화 출력 (JSON Schema)
- Ch 6. 툴 콜링 (Function Calling)
- Ch 7. 스트리밍 & UX
- Ch 8. 시스템 프롬프트 엔지니어링

계속해서 학습을 진행하세요!